In [0]:
df_date = spark.read.table("ecommerce.silver.slv_date")
df_date.show(5)

### Add is_weekend column
#### **using WithCOlumns**

In [0]:
from pyspark.sql.functions import dayofweek, col, when, regexp_replace, date_format

df_date = df_date.withColumns({
    "is_weekend" : when(dayofweek(col("Date")).isin(1,7), "1").otherwise("0"),
    "DateKey" : regexp_replace(col("Date"), "-", ""),
    "MonthName" : date_format(col("Date"), 'MMMM')
})

df_date.show(5)


In [0]:
desired_column_order = ['Date','Year','Day','Quarter','Week_Of_Year','_file_path','Injested_at','is_weekend','DateKey']

df_date = df_date.select(desired_column_order)

In [0]:
df_date.show(5)

####  Write as Gold DF 


In [0]:
df_date.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("s3://sj-dbr-demo-proj/gold_data/dim_date/")

spark.sql("""
          Create table if not exists ecommerce.gold.dim_date
          using delta
          location 's3://sj-dbr-demo-proj/gold_data/dim_date/'
          """)

In [0]:
dbutils.notebook.exit('SUCCESS')